<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/crewai_multiagent_level_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# workshop: Create Multi Agents to Research and Write an Article

In this workshop, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

https://huggingface.co/spaces/decodingdatascience/fromprompttopost

In [13]:
#install packages
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [3]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.



In [14]:
#setting the model
import os

#os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4.1-nano-2025-04-14'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [3]:
# Retrieve API key from Colab secrets and set it in environment variables
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('openai')

In [6]:
from crewai_tools import (
    DirectoryReadTool,
    FileReadTool,
    SerperDevTool,
    WebsiteSearchTool
)

In [8]:
# Set up API keys - enter from https://serper.dev/billing
os.environ["SERPER_API_KEY"] = "218873db9b73d345d3083e2f1f29cb74b14d4ddb" # serper.dev API key

# Instantiate tools
docs_tool = DirectoryReadTool(directory='./data')
file_tool = FileReadTool()
search_tool = SerperDevTool()
web_rag_tool = WebsiteSearchTool()

In [9]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic."    ,
    tools=[search_tool, web_rag_tool,docs_tool,file_tool],
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [10]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic in 1500 words: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [11]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [12]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [18]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.1500 words",
    agent=writer,
)

### Task: Edit

In [19]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.

In [20]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    planning=True,
    verbose=4
)

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

## Running the Crew
## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [21]:
topic = "Muse Spark meta"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Muse Spark meta.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...


Thought: To develop an informed and comprehensive content plan on Muse Spark meta, I need to gather the latest trends, key players, and recent news about Muse Spark meta. Additionally, understanding the target audience's interests and pain points is crucial for tailoring the content effectively. I will start by searching the internet for recent articles, news, and discussions related to Muse Spark meta to collect relevant insights.

Action: Search the internet

Action Input: {"search_query": "latest trends and news on Muse Spark meta"}

 


Search results: Title: Is Mark Zuckerberg's Meta AI getting too smart? - Fox News
Link: https://www.foxnews.com/tech/mark-zuckerbergs-meta-ai-getting-smart
Snippet: Meta's new AI model, Muse Spark, powers smarter, faster, more personal experiences across the apps you already use. Jesse Watson ...
---
Title: Meta's Muse Spark Model: Big Improvements Reinvigorate Investors
Link: https://finance.yahoo.com/markets/stocks/articles/metas-muse-spark-model-big-151500205.html
Snippet: Meta Platforms' Muse Spark AI model just injected investors with a much-needed boost of confidence, with shares spiking after its release.
---
Title: Introducing Muse Spark: Meta's Most Powerful Model Yet
Link: https://about.fb.com/news/2026/04/introducing-muse-spark-meta-superintelligence-labs/
Snippet: Muse Spark powers a smarter and faster Meta AI assistant, and will be rolling out to WhatsApp, Instagram, Facebook, Messenger, ...
---
Title: Introducing Muse Spark: Scaling Towards Personal Superintelligence


Thought: I now can give a great answer

Final Answer:

### Content Plan for Blog Article on Muse Spark Meta

---

#### **Title:**  
Unveiling Muse Spark: Meta's Next-Generation AI Model and Its Implications

---

### **1. Audience Analysis**

**Target Audience:**  
- Tech enthusiasts and AI researchers interested in the latest developments in artificial intelligence.  
- Business leaders and investors seeking insights into Meta's AI innovations and their market impact.  
- Developers and product managers exploring integration opportunities with AI models.  
- General readers with an interest in social media technology and AI ethics.

**Interests and Pain Points:**  
- Understanding advanced AI capabilities and how Muse Spark differs from previous models.  
- Evaluating the practical applications and limitations of Muse Spark.  
- Concerns about data privacy, security, and the reliability of AI-generated advice.  
- Keeping abreast of Meta's strategic direction in AI and social media.  

Final Answer:

# Unveiling Muse Spark: Meta's Next-Generation AI Model and Its Implications

The rapid evolution of artificial intelligence (AI) continues to reshape the technological landscape, influencing industries from healthcare to entertainment. Among the giants leading this charge is Meta, formerly Facebook, which has consistently pushed the boundaries of AI research to enhance social connectivity and user experience. As of 2024, Meta has introduced its most ambitious AI model to date: **Muse Spark**. This groundbreaking development not only exemplifies Meta’s commitment to advancing AI but also signals a potential shift towards a new era of multimodal, multi-agent, and ethically conscious artificial intelligence. In this article, we delve into what Muse Spark is, its innovative features, recent developments, and the broader implications for the industry and society.

---

## What Is Muse Spark? A Leap Into the Future of AI

Muse Spark is Meta’s latest flagship AI model, designe

In [22]:
from IPython.display import Markdown
Markdown(result)

# Unveiling Muse Spark: Meta's Next-Generation AI Model and Its Implications

The rapid evolution of artificial intelligence (AI) continues to reshape the technological landscape, influencing industries from healthcare to entertainment. Among the giants leading this charge is Meta, formerly Facebook, which has consistently pushed the boundaries of AI research to enhance social connectivity and user experience. As of 2024, Meta has introduced its most ambitious AI model to date: **Muse Spark**. This groundbreaking development not only exemplifies Meta’s commitment to advancing AI but also signals a potential shift toward a new era of multimodal, multi-agent, and ethically conscious artificial intelligence. In this article, we explore what Muse Spark is, its innovative features, recent developments, and the broader implications for the industry and society.

---

## What Is Muse Spark? A Leap Into the Future of AI

Muse Spark is Meta’s latest flagship AI model, designed to be a **multimodal reasoning engine** capable of integrating and processing diverse types of data—text, images, videos, and even contextual cues—simultaneously. Unlike earlier models that focused primarily on singular modalities, such as language-only models like GPT, Muse Spark embodies a **holistic approach** that enables it to understand and generate complex, nuanced outputs across multiple formats.

Built upon a **state-of-the-art architecture**, Muse Spark supports **tool-use capabilities**, allowing it to leverage external tools and databases to enhance reasoning and problem-solving skills. This feature mirrors human behavior, where references and analytical tools are consulted to make informed decisions. Additionally, its **visual chain of thought** functionality enables the model to explain its reasoning process step-by-step, promoting transparency and interpretability—crucial factors when deploying AI in sensitive or high-stakes domains.

Technologically, Muse Spark is underpinned by advanced neural networks trained on vast, diverse datasets from Meta’s ecosystem and beyond. Its **modular architecture** facilitates **multi-agent orchestration**, enabling different AI components to collaborate seamlessly—much like a team of specialists working together toward a common goal. This design aims to improve versatility, adaptability, and contextual understanding across a wide range of applications.

---

## Key Trends and Innovations: The New Paradigm in AI

### Multimodal Capabilities and Contextual Understanding

One of Muse Spark’s most significant features is its **multimodal reasoning** ability. This allows the AI to analyze and synthesize information from visual and textual sources simultaneously, effectively bridging the gap between images, videos, and language. For example, in social media contexts, Muse Spark could interpret a shared photo alongside user comments, extracting meaningful insights or generating pertinent responses.

This multimodal approach aligns with broader industry trends striving for **more human-like AI** that comprehends context in a multi-sensory manner. It paves the way for **personalized user experiences**, including intelligent content moderation, virtual assistants, and immersive digital interactions that respond to multifaceted inputs.

### Integration Within Meta’s Ecosystem

Meta’s extensive ecosystem, which includes platforms like Facebook, Instagram, WhatsApp, and Messenger, benefits from Muse Spark’s deep integration. This enables smarter content curation, more effective moderation, and intuitive user interactions. For instance, AI-powered features could automatically flag harmful content, assist users in composing messages, or generate multimedia content, enhancing both safety and engagement across services.

### The Shift Toward Personal Superintelligence

Meta envisions Muse Spark as a step toward **personal superintelligence**—AI systems capable of assisting individuals in a highly personalized manner across their daily digital activities. This aligns with Meta’s broader goal of creating **adaptive, intelligent interfaces** that anticipate user needs, facilitate seamless communication, and support applications in health, education, and beyond.

### Industry Comparisons and Competitive Landscape

In the broader AI arena, Muse Spark positions Meta as a formidable competitor alongside Google DeepMind, OpenAI, and Microsoft. While models like GPT-4 have demonstrated impressive language capabilities, Muse Spark’s emphasis on **multimodal and multi-agent functionalities** pushes innovation further, emphasizing **versatility and contextual depth**. This approach reflects Meta’s strategy of developing more **integrative and socially aware AI systems** suited for the complex, multi-format nature of social media content.

---

## Recent News and Developments: What’s Been Happening?

### Launch Announcements and Strategic Goals

Meta officially announced Muse Spark in early 2024, highlighting it as a **cornerstone of their AI ambitions**. The company emphasized its focus on **building responsible, ethical AI** capable of supporting a wide array of applications—from social media management to health insights. According to Meta’s statements, Muse Spark aims to **accelerate platform innovation** while upholding commitments to user privacy and ethical standards.

### Market and Investor Reactions

Following the announcement, Meta’s stock experienced a positive uptick, reflecting investor confidence in its AI trajectory. Industry analysts see Muse Spark as a **potential game-changer**, especially given Meta’s vast user base and data resources. Pilot programs, such as AI-driven content moderation and customer support bots, have already been deployed, generating both enthusiasm and cautious optimism regarding its capabilities and societal impact.

### Challenges and Ethical Concerns

Despite its promising features, Muse Spark faces scrutiny over issues like **reliability** and **ethical considerations**. Critics have raised concerns about **misinformation**, especially since AI can generate highly realistic visual and textual content. Data privacy remains a critical issue, as multimodal AI systems require access to large amounts of personal data, raising questions about consent and security.

Instances where AI models provide **unreliable health advice** highlight the importance of **robust oversight** and ethical safeguards. Meta acknowledges these challenges, emphasizing a commitment to **responsible AI development**, transparency, and ongoing monitoring to minimize risks and promote public trust.

---

## The Key Players and Stakeholders

### Meta’s Leadership and Research Teams

Meta’s AI advancements are driven by specialized teams such as the **Superintelligence Labs**, led by experts like **Alexandr Wang**. These teams focus on translating cutting-edge research into practical applications, ensuring that Muse Spark remains at the forefront of innovation while aligning with societal values and ethical standards.

### Industry Collaborators and Partners

Meta collaborates with academic institutions, industry partners, and regulatory bodies to foster **open research** and establish standards for ethical AI deployment. Such partnerships aim to promote transparency, societal benefits, and responsible innovation, ensuring that AI developments serve the broader public interest.

### Competitors and Industry Dynamics

Within the competitive landscape, Meta’s AI efforts are set against initiatives by Google DeepMind, OpenAI, and Microsoft. Each organization pursues different strategies—some focus on language models, others on multimodal systems—highlighting the race to achieve more **powerful, ethical, and scalable AI** solutions. This dynamic underscores the importance of **speed, collaboration, and responsible innovation** in shaping the future of AI.

### Ethical Organizations and Regulators

Organizations such as the **Partnership on AI** and governmental regulators play a vital role in establishing **ethical standards** and **policy frameworks**. Their involvement reflects a growing consensus on the need for **accountability and transparency** in deploying advanced AI systems like Muse Spark, ensuring societal safeguards keep pace with technological progress.

---

## Implications and Future Outlook: What’s Next?

### Transformative Applications Across Industries

Muse Spark’s multimodal capabilities could revolutionize numerous sectors. In **social media**, it can enhance content recommendation, moderation, and user engagement. In **healthcare**, it might assist diagnostics by analyzing images alongside patient data. **Customer service** could become more intuitive and efficient through AI systems that understand complex, multi-format queries and provide personalized responses.

### Ethical and Societal Considerations

The rapid advancement of AI also raises significant ethical questions. Risks related to **bias**, **misinformation**, and **misuse** are amplified as these systems become more sophisticated. Meta’s emphasis on **transparency**, **accountability**, and **ethical design** will be critical in addressing these concerns and ensuring AI benefits society without unintended harm.

### Future Trajectory of Meta’s AI Strategy

Looking ahead, Meta appears committed to **responsible innovation**, balancing technological breakthroughs with societal impacts. The development of **personalized, multimodal AI** like Muse Spark hints at a future where social media platforms are deeply integrated with intelligent, adaptive systems that foster richer human-AI collaboration and support a broader ecosystem of health, education, and social engagement tools.

### Industry Standards and User Experience

As Muse Spark is integrated into Meta’s platforms, it is poised to influence **industry standards** for multimodal and multi-agent AI systems. Its success could set benchmarks for **privacy protocols, user experience, and ethical deployment**, shaping the broader AI landscape and fostering a more responsible, user-centered approach to technological innovation.

---

## Conclusion: A New Dawn in AI with Muse Spark

Muse Spark marks a significant milestone in Meta’s AI journey, embodying a convergence of technological innovation, ethical responsibility, and strategic vision. Its multimodal, multi-agent architecture has the potential to redefine how AI interacts with humans and data, especially within social media ecosystems. While challenges related to privacy, bias, and misinformation remain, the potential benefits—more intuitive, personalized, and versatile AI systems—are profound.

As AI continues to evolve rapidly, collaboration among researchers, industry leaders, policymakers, and society at large will be essential. Meta’s Muse Spark exemplifies both the promise and the responsibility inherent in advancing AI. It offers a glimpse into a future where technology enhances human experiences responsibly and ethically, fostering deeper connections and smarter digital environments.

**Stay informed about Meta’s AI innovations and participate in shaping a responsible AI future. Your engagement and insights are vital as we navigate this transformative era together.** Whether you’re a tech enthusiast, developer, or concerned citizen, your voice can help steer AI development toward beneficial and equitable outcomes.

---

*This article synthesizes official Meta sources, industry news, and expert insights to provide an objective and comprehensive perspective on Muse Spark and its broader societal implications.*

# Deploy this crew in gradio

In [1]:
pip install gradio

In [4]:
import gradio as gr
from crewai import Agent, Task, Crew

# Define Agents
researcher = Agent(
    role="Senior Research Specialist",
    goal="Uncover intricate insights into the given topic",
    backstory="A seasoned expert in extracting valuable information and providing detailed analysis on any given subject.",
    allow_delegation=False
)

writer = Agent(
    role="Content Writer",
    goal="Craft engaging and informative articles",
    backstory="An experienced content writer who can turn complex research into easy-to-understand and engaging articles.",
    allow_delegation=False
)

editor = Agent(
    role="Content Editor",
    goal="Ensure articles are polished, coherent, and engaging",
    backstory="An expert editor with a sharp eye for clarity, grammar, and flow, capable of turning drafts into publication-ready articles.",
    allow_delegation=False
)

# Function to setup tasks and run CrewAI
def run_crewai_article_writer(topic):
    # Define tasks
    research_task = Task(
        description=f"Conduct in-depth research on the topic: {topic} and provide a structured summary.",
        expected_output="A structured and detailed research summary covering important aspects, recent developments, and insights related to the topic.",
        agent=researcher
    )

    writing_task = Task(
        description=f"Using the research, write a full-length article on the topic: {topic} in 1500 words give various sub section",
        expected_output="An engaging, detailed, informative, and coherent article suitable for online publication.",
        agent=writer
    )

    editing_task = Task(
        description=f"Review and edit the article on {topic} for clarity, grammar, flow, and engagement.",
        expected_output="A polished, clear, and professionally edited final version of the article.",
        agent=editor
    )

    # Setup Crew
    crew = Crew(
        agents=[researcher, writer, editor],
        tasks=[research_task, writing_task, editing_task],
        verbose=True  # Set to False if you want less console output
    )

    # Execute
    result = crew.kickoff()
    return result

# Gradio Interface
def generate_article(topic):
    return run_crewai_article_writer(topic)

iface = gr.Interface(
    fn=generate_article,
    inputs=gr.Textbox(lines=2, placeholder="Enter a topic for the article..."),
    outputs=gr.Textbox(lines=20, label="Generated Article"),
    title="AI Agent: Prompt to Post",
    description="Enter a topic and let CrewAI agents research, write, and edit a complete article!"
)

if __name__ == "__main__":
    iface.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f32acd2d3790bf834e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
